# 📐 Segmentação de Trincas por Limiarização de Otsu

**Disciplina:** IA Aplicada à Construção Civil  
**Módulo:** Visão Computacional — Segmentação e Avaliação de Modelos

---

## Contexto

Segmentar uma trinca significa decidir, **pixel a pixel**, o que é trinca e o que é fundo. O resultado é uma **máscara binária**: branco (255) onde há trinca, preto (0) onde não há.

Este é o problema mais fundamental de visão computacional aplicada à inspeção: antes de medir, classificar ou reportar uma patologia, é preciso saber *onde ela está* na imagem.

### Limiarização global vs. Otsu

A forma mais simples de segmentar é a **limiarização**: pixels abaixo de um valor $T$ (limiar) vão para uma classe, acima para outra.

$$\text{mask}(x,y) = \begin{cases} 255 & \text{se } I(x,y) < T \\ 0 & \text{caso contrário} \end{cases}$$

O problema é: **qual valor de $T$ usar?** Uma trinca em ambiente interno tem brilho diferente de uma trinca fotografada ao sol. Escolher $T$ manualmente para cada imagem é inviável em lote.

**Otsu (1979)** resolve isso automaticamente: ele encontra o limiar $T^*$ que **maximiza a variância entre classes** (trinca vs. fundo), ou equivalentemente, minimiza a variância dentro de cada classe. A busca é feita diretamente no histograma — custo $O(256)$, independente da resolução da imagem.

$$T^* = \arg\max_T \; \omega_0(T)\,\omega_1(T)\,\bigl[\mu_0(T) - \mu_1(T)\bigr]^2$$

onde $\omega_i$ é a proporção de pixels e $\mu_i$ é a média de intensidade de cada classe.

### Quando Otsu funciona bem — e quando falha

| Condição | Otsu funciona? | Por quê |
|----------|---------------|--------|
| Histograma bimodal claro (dois picos) | ✅ | A separação entre classes é natural |
| Fundo uniforme, trinca escura e fina | ✅ | Contraste alto entre classes |
| Iluminação muito irregular | ⚠️ | Histograma unimodal → limiar ruim |
| Trinca clara em fundo escuro | ⚠️ | Precisa inverter a máscara |
| Múltiplos materiais com tons similares | ❌ | Histograma multimodal confunde Otsu |

> 💡 **Pré-processamento importa:** aplicar GaussianBlur antes do Otsu remove ruído de alta frequência que criaria picos espúrios no histograma. Os módulos anteriores (CLAHE, remoção de artefatos) melhoram diretamente a qualidade do Otsu.

---

## Como avaliamos a segmentação?

Para saber se a máscara gerada é boa, comparamos com um **Ground Truth (GT)** — uma máscara criada manualmente por um especialista, considerada a referência correta.

As duas métricas usadas neste notebook:

### IoU — Intersection over Union

$$\text{IoU} = \frac{|\text{predição} \cap \text{GT}|}{|\text{predição} \cup \text{GT}|}$$

Mede a sobreposição entre a máscara predita e o GT. Valor entre 0 e 1:
- **1.0** → predição perfeita
- **0.5** → critério mínimo comum em benchmarks de segmentação (COCO)
- **< 0.3** → segmentação pobre

É a métrica preferida para segmentação porque **penaliza falsos positivos e falsos negativos** de forma equilibrada.

### Acurácia pixel a pixel

$$\text{Acc} = \frac{\text{pixels corretos}}{\text{total de pixels}}$$

Parece intuitiva, mas tem um problema sério em imagens de trincas: o fundo (pixels pretos) domina a imagem. Um modelo que prevê **tudo como fundo** pode ter 98% de acurácia sem detectar nenhuma trinca. Por isso, **IoU é mais confiável** para patologias — a acurácia é apresentada aqui apenas como referência comparativa.

---

## Estrutura deste notebook

| Seção | O que você vai fazer |
|-------|----------------------|
| 1 | Testar o ambiente com imagem sintética (sem Drive) |
| 2 | Montar o Google Drive e configurar caminhos |
| 3 | Definir o pipeline de segmentação e avaliação |
| 4 | Processar todas as imagens em lote |
| 5 | Visualizar resultados, histogramas e métricas |
| 6 | Calibrar com grade de limiares e blur |

> 💡 **Recomendação:** execute a Seção 1 primeiro — ela não depende de nenhum arquivo externo e mostra o histograma bimodal que o Otsu precisa enxergar para funcionar bem.

---
## Seção 1 — Teste com imagem sintética

Criamos uma imagem que simula condições típicas de inspeção:

- **Fundo cinza com ruído** → concreto
- **Trinca escura e fina** → o que queremos segmentar
- **Ground Truth manual** → retângulo que simula anotação de especialista

**O que observar:**
1. O histograma deve ter **dois picos** (bimodal) — um para o fundo, um para a trinca
2. O limiar de Otsu deve cair **entre os dois picos**
3. A máscara Otsu deve se aproximar do Ground Truth → IoU alto

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# ── Criar imagem sintética 400×600 (BGR)
h, w = 400, 600
img_sint = np.full((h, w, 3), 160, dtype=np.uint8)

# Ruído gaussiano (textura do concreto)
ruido = np.random.normal(0, 12, img_sint.shape).astype(np.int16)
img_sint = np.clip(img_sint.astype(np.int16) + ruido, 0, 255).astype(np.uint8)

# Trinca escura diagonal + ramificação
pts = np.array([[60, 30], [180, 160], [310, 260], [490, 370]], dtype=np.int32)
for i in range(len(pts) - 1):
    cv2.line(img_sint, tuple(pts[i]), tuple(pts[i+1]), (28, 28, 28), 2)
cv2.line(img_sint, (220, 195), (265, 245), (35, 35, 35), 1)  # ramificação

# ── Ground Truth manual (máscara que um especialista desenharia)
gt = np.zeros((h, w), dtype=np.uint8)
for i in range(len(pts) - 1):
    cv2.line(gt, tuple(pts[i]), tuple(pts[i+1]), 255, 5)   # ligeiramente mais grosso
cv2.line(gt, (220, 195), (265, 245), 255, 3)

# ── Pipeline Otsu
gray = cv2.cvtColor(img_sint, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)
T_otsu, otsu_mask = cv2.threshold(blur, 0, 255,
                                   cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# ── Métricas
intersection = np.logical_and(gt == 255, otsu_mask == 255)
union        = np.logical_or (gt == 255, otsu_mask == 255)
iou      = np.sum(intersection) / np.sum(union) if np.sum(union) > 0 else 0
accuracy = np.sum(gt == otsu_mask) / gt.size

# ── Visualização principal
fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
dados = [
    (img_sint, 'Original (sintética)', None),
    (gt,       'Ground Truth (manual)', 'gray'),
    (otsu_mask,f'Segmentação Otsu\n(T* = {T_otsu:.0f})', 'gray'),
    (cv2.absdiff(gt, otsu_mask), 'Diferença GT × Otsu\n(branco = discordância)', 'Reds'),
]
for ax, (im, t, cmap) in zip(eixos, dados):
    ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
    ax.set_title(t, fontsize=9)
    ax.axis('off')

plt.suptitle(f'✅ Teste sintético — IoU: {iou:.4f}  |  Acurácia: {accuracy:.4f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Histograma com limiar de Otsu
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(gray.ravel(), bins=256, range=(0, 255), color='steelblue', alpha=0.8)
ax1.axvline(T_otsu, color='red', linewidth=2, label=f'T* Otsu = {T_otsu:.0f}')
ax1.set_title('Histograma — imagem cinza')
ax1.set_xlabel('Intensidade'); ax1.set_ylabel('Frequência')
ax1.legend()

ax2.hist(blur.ravel(), bins=256, range=(0, 255), color='darkorange', alpha=0.8)
ax2.axvline(T_otsu, color='red', linewidth=2, label=f'T* Otsu = {T_otsu:.0f}')
ax2.set_title('Histograma — após GaussianBlur')
ax2.set_xlabel('Intensidade')
ax2.legend()

plt.suptitle('Histogramas — note os dois picos (bimodal) que o Otsu precisa separar',
             fontsize=11)
plt.tight_layout()
plt.show()

print(f'Limiar de Otsu encontrado : T* = {T_otsu:.0f}')
print(f'IoU                       : {iou:.4f}')
print(f'Acurácia pixel a pixel    : {accuracy:.4f}')
print('\n✅ Ambiente OK — pode prosseguir para a Seção 2.')

---
## Seção 2 — Montar o Google Drive e configurar

> ⚠️ **Pasta compartilhada?** Se `fotos_obra` não estiver em *Meu Drive*:  
> *Drive → clique direito → Organizar → Adicionar atalho ao Drive*

> 📌 **Ground Truth real:** neste notebook o GT é gerado por limiarização binária simples (aproximação). Em um projeto real, o GT deve ser criado no **Label Studio** com anotações de especialista e exportado como máscara PNG.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive montado.')

In [ ]:
from pathlib import Path
from tqdm.notebook import tqdm
import pandas as pd

# ── Caminhos
PASTA_ENTRADA = Path('/content/drive/MyDrive/IPOS_02/fotos_obra')
PASTA_GT      = Path('/content/drive/MyDrive/IPOS_02/fotos_obra_gt')   # máscaras GT (opcional)
PASTA_SAIDA   = Path('/content/drive/MyDrive/IPOS_02/fotos_obra_otsu')
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# ── Parâmetros
BLUR_KSIZE  = 5     # tamanho do kernel GaussianBlur (ímpar, 3–11)
GT_LIMIAR   = 120   # limiar fixo para gerar GT aproximado (usado se não houver GT manual)
LARGURA_MAX = 1200  # 0 = sem resize

# ── Extensões
EXTENSOES = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

# ── Verificar
if not PASTA_ENTRADA.exists():
    print(f'⛔ Pasta não encontrada: {PASTA_ENTRADA}')
else:
    arquivos = sorted([p for p in PASTA_ENTRADA.iterdir() if p.suffix.lower() in EXTENSOES])
    tem_gt   = PASTA_GT.exists()
    print(f'📂 Entrada        : {PASTA_ENTRADA}')
    print(f'📁 Ground Truth   : {PASTA_GT}  {"✅ encontrada" if tem_gt else "⚠️  não encontrada — será usado GT aproximado"}')
    print(f'📁 Saída          : {PASTA_SAIDA}')
    print(f'⚙️  blur_ksize     : {BLUR_KSIZE}')
    print(f'⚙️  gt_limiar      : {GT_LIMIAR} (usado apenas se não houver GT manual)')
    print(f'🖼️  {len(arquivos)} imagem(ns) encontrada(s)')

---
## Seção 3 — Pipeline de segmentação e avaliação

### Fluxo completo

```
Imagem colorida (BGR)
        │
        ▼
  cvtColor → GRAY          ← elimina informação de cor (trinca é escura em qualquer canal)
        │
        ▼
  GaussianBlur (ksize)     ← suaviza ruído de alta frequência → histograma mais bimodal
        │
        ▼
  THRESH_BINARY_INV        ← inverte: trinca escura vira branco (255)
  + THRESH_OTSU            ← encontra T* automaticamente no histograma
        │
        ▼
  Máscara Otsu
        │
        ├──────────────────────────────────────────────────────
        │                Ground Truth disponível?
        │         SIM                           NÃO
        │          │                             │
        │   Carrega PNG de GT          Gera GT aproximado
        │   (Label Studio export)      (THRESH_BINARY_INV, T fixo)
        │          │                             │
        └──────────┴─────────────────────────────┘
                   │
                   ▼
        Calcular IoU e Acurácia
                   │
                   ▼
        Salvar: máscara + comparativo + métricas CSV
```

### Por que `THRESH_BINARY_INV`?

Trincas em concreto são **mais escuras** que o fundo. O Otsu padrão (`THRESH_BINARY`) coloca pixels **acima** do limiar em branco. Invertendo, colocamos pixels **abaixo** do limiar (os mais escuros — as trincas) em branco. Resultado: trinca = branco, fundo = preto, que é a convenção de máscara usada em todo o pipeline deste curso.

In [ ]:
def gerar_gt_aproximado(gray, limiar=120):
    """GT simples por limiar fixo — substituto enquanto não há anotação manual."""
    _, gt = cv2.threshold(gray, limiar, 255, cv2.THRESH_BINARY_INV)
    return gt


def carregar_gt_manual(pasta_gt, nome_stem):
    """Tenta carregar PNG de GT com mesmo nome da imagem."""
    for ext in ('.png', '.jpg', '.bmp'):
        caminho = pasta_gt / f'{nome_stem}{ext}'
        if caminho.exists():
            gt = cv2.imread(str(caminho), cv2.IMREAD_GRAYSCALE)
            if gt is not None:
                return gt
    return None


def segmentar_otsu(img_bgr, blur_ksize=5):
    """Converte para cinza, aplica blur e Otsu invertido."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    ksize = max(3, blur_ksize | 1)   # garante ímpar >= 3
    blur  = cv2.GaussianBlur(gray, (ksize, ksize), 0)
    T, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return gray, blur, mask, T


def calcular_metricas(gt, pred):
    """Calcula IoU e acurácia entre duas máscaras binárias."""
    # Garante binarização caso GT não seja 0/255 puro
    gt_b   = (gt   > 127).astype(bool)
    pred_b = (pred > 127).astype(bool)

    inter = np.logical_and(gt_b, pred_b).sum()
    uniao = np.logical_or (gt_b, pred_b).sum()
    iou   = inter / uniao if uniao > 0 else 0.0
    acc   = (gt_b == pred_b).sum() / gt_b.size

    tp = inter
    fp = np.logical_and(~gt_b, pred_b).sum()
    fn = np.logical_and(gt_b, ~pred_b).sum()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)

    return {'iou': round(iou, 4), 'accuracy': round(acc, 4),
            'precision': round(precision, 4), 'recall': round(recall, 4),
            'f1': round(f1, 4)}


print('✅ Funções do pipeline definidas.')

---
## Seção 4 — Processamento em lote

Para cada imagem são gerados 4 arquivos na pasta de saída:

```
fotos_obra_otsu/
  nome_foto_gt.png          ← Ground Truth usado (manual ou aproximado)
  nome_foto_otsu.png        ← máscara Otsu
  nome_foto_diff.png        ← mapa de discordância GT × Otsu
  nome_foto_comparativo.png ← painel 4 vistas
  metricas_otsu.csv         ← tabela consolidada de IoU, acurácia, F1
```

In [ ]:
todas_metricas = []
erros          = []

for arq in tqdm(arquivos, desc='Segmentando'):
    try:
        img = cv2.imread(str(arq))
        assert img is not None

        if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
            r   = LARGURA_MAX / img.shape[1]
            img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0] * r)),
                             interpolation=cv2.INTER_AREA)

        gray, blur_img, otsu_mask, T = segmentar_otsu(img, BLUR_KSIZE)

        # Ground Truth
        gt_manual = carregar_gt_manual(PASTA_GT, arq.stem) if PASTA_GT.exists() else None
        if gt_manual is not None:
            gt       = gt_manual
            gt_fonte = 'manual'
        else:
            gt       = gerar_gt_aproximado(gray, GT_LIMIAR)
            gt_fonte = 'aproximado'

        metricas = calcular_metricas(gt, otsu_mask)
        metricas.update({'arquivo': arq.name, 'T_otsu': int(T), 'gt_fonte': gt_fonte})
        todas_metricas.append(metricas)

        # Mapa de diferença
        diff = cv2.absdiff(gt, otsu_mask)

        # Painel comparativo
        def to_bgr(m): return cv2.cvtColor(m, cv2.COLOR_GRAY2BGR)
        painel_top = np.hstack([img, to_bgr(gt)])
        painel_bot = np.hstack([to_bgr(otsu_mask), to_bgr(diff)])
        painel     = np.vstack([painel_top, painel_bot])

        # Legenda
        cv2.rectangle(painel, (10, 10), (680, 62), (255, 255, 255), -1)
        cv2.putText(painel,
                    f'T*={T:.0f}  IoU={metricas["iou"]:.3f}  F1={metricas["f1"]:.3f}  GT={gt_fonte}',
                    (18, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (20, 20, 20), 2, cv2.LINE_AA)

        base = arq.stem
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_gt.png'),          gt)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_otsu.png'),        otsu_mask)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_diff.png'),        diff)
        cv2.imwrite(str(PASTA_SAIDA / f'{base}_comparativo.png'), painel)

    except Exception as e:
        erros.append((arq.name, str(e)))

# Relatório
df = pd.DataFrame(todas_metricas)
caminho_csv = PASTA_SAIDA / 'metricas_otsu.csv'
df.to_csv(str(caminho_csv), index=False)

print(f'\n✅ {len(todas_metricas)} imagem(ns) processada(s)')
print(f'   IoU médio : {df["iou"].mean():.4f}')
print(f'   F1 médio  : {df["f1"].mean():.4f}')
print(f'   Relatório : {caminho_csv}')
if erros:
    print('\n⚠️  Erros:')
    for n, m in erros: print(f'   {n}: {m}')

---
## Seção 5 — Visualização, histogramas e métricas

### 5a. Painel de quatro vistas

| Quadrante | O que analisar |
|-----------|----------------|
| **Original** | Referência visual |
| **Ground Truth** | Se for aproximado (limiar fixo), pode incluir falsos positivos em regiões escuras |
| **Otsu** | Compare com GT — deve capturar as mesmas regiões escuras |
| **Diferença** | Branco = discordância. Concentrada na trinca → erro de precisão. Espalhada pelo fundo → falsos positivos |

In [ ]:
MAX_EXIBIR = 4

for arq in arquivos[:MAX_EXIBIR]:
    img = cv2.imread(str(arq))
    if img is None: continue
    if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
        r = LARGURA_MAX / img.shape[1]
        img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0]*r)), interpolation=cv2.INTER_AREA)

    gray, blur_img, otsu_mask, T = segmentar_otsu(img, BLUR_KSIZE)
    gt_manual = carregar_gt_manual(PASTA_GT, arq.stem) if PASTA_GT.exists() else None
    gt = gt_manual if gt_manual is not None else gerar_gt_aproximado(gray, GT_LIMIAR)
    metricas = calcular_metricas(gt, otsu_mask)
    diff = cv2.absdiff(gt, otsu_mask)

    fig, eixos = plt.subplots(1, 4, figsize=(22, 5))
    dados = [
        (img,       'Original',                None),
        (gt,        'Ground Truth',            'gray'),
        (otsu_mask, f'Otsu  T*={T:.0f}',       'gray'),
        (diff,      'Discordância GT × Otsu',  'Reds'),
    ]
    for ax, (im, t, cmap) in zip(eixos, dados):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB) if cmap is None else im, cmap=cmap)
        ax.set_title(t, fontsize=9)
        ax.axis('off')

    plt.suptitle(
        f'{arq.name}   IoU={metricas["iou"]:.4f}  '
        f'Acc={metricas["accuracy"]:.4f}  '
        f'F1={metricas["f1"]:.4f}  '
        f'Precision={metricas["precision"]:.4f}  '
        f'Recall={metricas["recall"]:.4f}',
        fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

### 5b. Histograma com limiar de Otsu

Visualiza onde o Otsu cortou o histograma. **O ideal:** o limiar deve cair no **vale entre dois picos**. Se o histograma for unimodal (um só pico), o Otsu vai cortar em um ponto arbitrário e a segmentação será ruim — sinal de que é necessário pré-processar antes (CLAHE, remoção de artefatos).

In [ ]:
if arquivos:
    img = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img.shape[1] > LARGURA_MAX:
        r = LARGURA_MAX / img.shape[1]
        img = cv2.resize(img, (LARGURA_MAX, int(img.shape[0]*r)), interpolation=cv2.INTER_AREA)

    gray, blur_img, otsu_mask, T = segmentar_otsu(img, BLUR_KSIZE)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

    ax1.hist(gray.ravel(), bins=256, range=(0, 255), color='steelblue', alpha=0.8)
    ax1.axvline(T, color='red', lw=2, label=f'T* Otsu = {T:.0f}')
    ax1.fill_betweenx([0, ax1.get_ylim()[1] if ax1.get_ylim()[1] > 0 else 1],
                       0, T, alpha=0.12, color='red', label='Trinca (< T*)')
    ax1.set_title(f'Histograma — cinza original\n{arquivos[0].name}', fontsize=10)
    ax1.set_xlabel('Intensidade (0–255)')
    ax1.set_ylabel('Frequência')
    ax1.legend()

    ax2.hist(blur_img.ravel(), bins=256, range=(0, 255), color='darkorange', alpha=0.8)
    ax2.axvline(T, color='red', lw=2, label=f'T* Otsu = {T:.0f}')
    ax2.fill_betweenx([0, ax2.get_ylim()[1] if ax2.get_ylim()[1] > 0 else 1],
                       0, T, alpha=0.12, color='red', label='Trinca (< T*)')
    ax2.set_title(f'Histograma — após GaussianBlur (k={BLUR_KSIZE})', fontsize=10)
    ax2.set_xlabel('Intensidade (0–255)')
    ax2.legend()

    plt.suptitle('Região em vermelho = pixels classificados como TRINCA pelo Otsu',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

### 5c. Relatório de métricas — ranking por IoU

IoU ≥ 0.5 é considerado o limiar mínimo aceitável em benchmarks de segmentação semântica (critério COCO). Abaixo disso, a segmentação não é confiável para uso em laudos ou treinamento de redes.

In [ ]:
df_exib = df[['arquivo', 'T_otsu', 'iou', 'f1', 'precision', 'recall', 'accuracy', 'gt_fonte']]\
            .sort_values('iou', ascending=False).reset_index(drop=True)
df_exib.index += 1

def colorir_iou(val):
    if   val >= 0.7 : return 'background-color: #d4edda'   # verde
    elif val >= 0.5 : return 'background-color: #fff3cd'   # amarelo
    else            : return 'background-color: #ffcccc'   # vermelho

display(df_exib.style.applymap(colorir_iou, subset=['iou']))

print('\n🟢 Verde   : IoU ≥ 0.7 — segmentação boa')
print('🟡 Amarelo : IoU 0.5–0.7 — aceitável, verificar visualmente')
print('🔴 Vermelho: IoU < 0.5 — segmentação ruim — calibrar parâmetros')
print(f'\nMédia IoU : {df["iou"].mean():.4f}')
print(f'Média F1  : {df["f1"].mean():.4f}')

---
## Seção 6 — Calibração: grade de blur × limiar manual

### O que calibrar e como

| Parâmetro | Efeito de aumentar | Quando aumentar |
|-----------|-------------------|----------------|
| `BLUR_KSIZE` | Histograma mais suave → Otsu mais estável | Imagem muito ruidosa |
| `GT_LIMIAR` | GT aproximado mais restritivo | Muitos falsos positivos no GT |

A grade abaixo varia o tamanho do blur (linhas) e mostra a máscara Otsu resultante. Compare visualmente com a imagem original para escolher o melhor `BLUR_KSIZE`.

In [ ]:
if arquivos:
    img_ref = cv2.imread(str(arquivos[0]))
    if LARGURA_MAX > 0 and img_ref.shape[1] > LARGURA_MAX:
        r       = LARGURA_MAX / img_ref.shape[1]
        img_ref = cv2.resize(img_ref, (LARGURA_MAX, int(img_ref.shape[0]*r)),
                             interpolation=cv2.INTER_AREA)

    blur_ksizes = [3, 5, 7, 11, 15]

    fig, eixos = plt.subplots(2, len(blur_ksizes) + 1, figsize=(4.5*(len(blur_ksizes)+1), 9))

    # Coluna 0 = original
    eixos[0][0].imshow(cv2.cvtColor(img_ref, cv2.COLOR_BGR2RGB))
    eixos[0][0].set_title('Original', fontsize=9)
    eixos[0][0].axis('off')
    eixos[1][0].axis('off')

    gray_ref = cv2.cvtColor(img_ref, cv2.COLOR_BGR2GRAY)

    for j, k in enumerate(blur_ksizes):
        blur_g = cv2.GaussianBlur(gray_ref, (k, k), 0)
        T_g, mask_g = cv2.threshold(blur_g, 0, 255,
                                     cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Linha 0 = máscara
        eixos[0][j+1].imshow(mask_g, cmap='gray')
        eixos[0][j+1].set_title(f'blur k={k}\nT*={T_g:.0f}', fontsize=9)
        eixos[0][j+1].axis('off')

        # Linha 1 = histograma com limiar
        eixos[1][j+1].hist(blur_g.ravel(), bins=100, range=(0,255),
                            color='steelblue', alpha=0.75)
        eixos[1][j+1].axvline(T_g, color='red', lw=1.5)
        eixos[1][j+1].set_title(f'Histograma (k={k})', fontsize=8)
        eixos[1][j+1].set_xlabel('Intensidade', fontsize=7)
        eixos[1][j+1].tick_params(labelsize=7)

    plt.suptitle(f'Grade de calibração — {arquivos[0].name}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    print('👆 Escolha o blur onde a máscara cobre a trinca sem excessos no fundo.')
    print('   Atualize BLUR_KSIZE na Seção 2 e reprocesse.')

---
## Próximos passos

O Otsu é o **baseline** — o ponto de partida para comparar qualquer método mais sofisticado. Se o IoU médio estiver abaixo de 0.5, o próximo passo não é afinar o Otsu, mas sim melhorar o pré-processamento:

```
IoU < 0.5 com Otsu?
        │
        ├── Aplicar CLAHE antes (módulo 03)        → histograma mais bimodal
        ├── Aplicar remoção de artefatos (módulo 07) → fundo mais uniforme
        └── Partir para segmentação adaptativa (CLAHE local + Otsu por bloco)

IoU ≥ 0.5 com Otsu?
        │
        └── Usar máscaras Otsu como pseudo-labels para treinar U-Net
            (iteração semi-supervisionada)
```

### Referências

- Otsu, N. (1979). *A Threshold Selection Method from Gray-Level Histograms*. IEEE Transactions on Systems, Man, and Cybernetics, 9(1), 62–66.
- Everingham, M. et al. (2010). *The Pascal VOC Challenge* — definição do critério IoU ≥ 0.5
- Lin, T-Y et al. (2014). *Microsoft COCO: Common Objects in Context* — benchmarks de segmentação
- OpenCV docs: [`cv2.threshold`](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html)
- ABNT NBR 6118:2014 — critérios de inspeção e classificação de fissuras em concreto
